In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
def load_words(path="names.txt"):
    with open(path, "r", encoding="utf-8") as f:
        return f.read().splitlines()

w = load_words()
words = w[:30]

In [3]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(stoi)

In [4]:
block_size = 3
X, Y = [], []

for w in words:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

In [5]:
class MakemoreMLP(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd=10, n_hidden=200):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, n_embd)
        self.fc1 = nn.Linear(block_size * n_embd, n_hidden)
        self.fc2 = nn.Linear(n_hidden, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)
        x = emb.view(emb.shape[0], -1)
        x = torch.tanh(self.fc1(x))
        logits = self.fc2(x)
        return logits

model = MakemoreMLP(vocab_size, block_size)

In [6]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for step in range(20000):
    logits = model(X)
    loss = F.cross_entropy(logits, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 2000 == 0:
        print(f"step {step}: loss = {loss.item():.4f}")

step 0: loss = 3.1307
step 2000: loss = 0.5509
step 4000: loss = 0.5504
step 6000: loss = 0.5504
step 8000: loss = 0.5503
step 10000: loss = 0.5504
step 12000: loss = 0.5503
step 14000: loss = 0.5502
step 16000: loss = 0.5502
step 18000: loss = 0.5503


In [7]:
@torch.no_grad()
def generate(model, max_length=20):
    out = []
    context = [0] * block_size

    for _ in range(max_length):
        x = torch.tensor([context])
        logits = model(x)
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item()

        if ix == 0:
            break

        out.append(itos[ix])
        context = context[1:] + [ix]

    return ''.join(out)

In [8]:
for _ in range(20):
    print(generate(model))

isabeth
victoria
harper
layla
madison
zoey
grace
charper
sophia
riley
sophia
harper
penelope
abigail
victoria
emila
camily
madison
ava
amelia
